# 116 — Herramientas tipadas y efectos laterales

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Una **herramienta tipada** es un contrato de tres partes: descripción semántica (cuándo
usarla y cuándo NO), **JSON Schema** de entrada (tipos, required, enums, rangos — se
valida ANTES de ejecutar) y contrato de salida/errores (el error estructurado es parte
de la interfaz: el agente lo observa y decide con él).

**Taxonomía de efectos:** pura/solo lectura → escritura reversible → escritura
irreversible → efecto externo distribuido. La clase de efecto determina qué controles
exige (reintento libre, registro, aprobación humana, auditoría).

### 🔁 Idempotencia y dry-run

**Idempotente:** `f(f(x)) = f(x)` — repetir la operación no multiplica el efecto
(`set_price(10)` sí; `add_units(+5)` no). Importa porque los agentes REINTENTAN y un
timeout no dice si el efecto se aplicó. Técnica estándar: **clave de idempotencia** —
el servidor registra las claves aplicadas y convierte duplicados en no-ops.

**Dry-run:** con `dry_run: true` la herramienta valida precondiciones y devuelve QUÉ
haría (diff/plan/costo) sin aplicar nada. Convierte un efecto irreversible en dos
pasos: uno observable y uno autorizado.

El laboratorio `agent` usa dos herramientas puras (`status`, `sum`): reintentables sin
riesgo — el caso base contra el que se mide todo lo demás.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("agent", seed=116)
show(result)


## Reflexión

1. Las dos herramientas del laboratorio son puras. ¿Qué tres mecanismos nuevos (schema,
   dry-run, idempotency_key) se vuelven obligatorios en cuanto una herramienta escribe
   estado, y qué riesgo concreto cubre cada uno?
2. ¿Por qué un timeout es el caso que separa "reintentar" de "reintentar con clave de
   idempotencia"? ¿Qué información NO te da un timeout?
3. Propón el error estructurado que debería devolver una herramienta `book_meeting`
   cuando la sala está ocupada, de modo que el siguiente thought pueda replantear sin
   intervención humana.